# T10 It4: stage cache and shared map FFT (3c)

Block `train/checks/stage_cache.py`, runs `Rich_Dump_Cached_executed_{cache_first, cache_reuse, cache_fft}` (driver `run_gpu.sh`): 5 official
photos x 27 queries x M = 10 (1,350 candidates).
- *cache_first*: fills the cache (canvas, the whole K = 50 coarse shortlist, DINO's score of each place) and matches as usual;
- *cache_reuse*: reads the cache, only fine matching runs;
- *cache_fft*: a new cache, with the map-window FFTs shared between the queries of a photo (patch of `correlate_everywhere`, no `src/` change).
Adopt only if the outputs are identical to the T6 dumps.

In [ ]:
import os, sys, glob
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
from train.checks.rich_dump import load_rich
R = ROOT / "tests/Test_Iteration_10/results/rich_dump"
rows = []
for tag in ["cache_first", "cache_reuse", "cache_fft"]:
    same_inl = same_place = same_z = tot = 0; maxdiff = 0.0
    for f in sorted(glob.glob(str(R / tag / "photo_*.npz"))):
        a = load_rich(f, with_matches=False); b = load_rich(ROOT / "tests/Test_Iteration_6/results/rich_dump/official" / os.path.basename(f), with_matches=False)
        for qa, qb in zip(a["queries"], b["queries"]):
            n = len(qa["inliers"]); tot += n
            same_inl += int((qa["inliers"] == qb["inliers"]).sum())
            same_place += n * int(np.allclose(qa["peaks"][:, 1:3], qb["peaks"][:, 1:3]))
            same_z += n * int(np.allclose(qa["z"], qb["z"], atol=1e-4))
            maxdiff = max(maxdiff, float(np.abs(qa["peaks"][:, 0] - qb["peaks"][:, 0]).max()))
    s = pd.read_csv(R / tag / "seconds.csv").seconds
    rows.append(dict(run=tag, candidates=tot, same_inliers=same_inl, same_places=same_place, same_dino_z=same_z, max_coarse_score_diff=maxdiff,
                     s_per_photo=round(s.mean(), 2), first_photo_s=round(s.iloc[0], 1)))
pd.DataFrame(rows)

**Read:** all three runs reproduce the T6 dumps exactly (every candidate's inliers, places and DINO z). Reusing the cache cuts a research run
from ~8.0 to ~4.7 s per photo (-41 %): matcher and selection experiments now cost only fine matching. Sharing the map FFT is identical but
saves only ~3 % on this GPU (cuFFT is fast here); adopted for the research runner (`SHARE_FFT=1`). On the Jetson the headless pipeline already
shares it (cuFFT fails there and the matrix-multiply FFT is slow, so the saving is larger there).